### This book shows how we fill library, and here we will collect results 

What should our tensors include?
Features which describe size of catenary curves, transmissions and phase shift.    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import meep as mp
import math
from IPython.display import clear_output

import conf

We count once flux_obj  for the substrate

In [ ]:
mp.quiet()
pml_layers = [mp.PML(thickness=conf.in_pml, direction=mp.X)]
cell = mp.Vector3(conf.cell_X, conf.cell_Y, 0)
symmetries = [mp.Mirror(mp.Y)]
geometry = [mp.Block(
    material=conf.in_material,
    size=mp.Vector3(conf.in_sub, mp.inf, conf.cell_Z),
    center=conf.sub_center,
)]
sources = [mp.Source(
    mp.ContinuousSource(conf.inc_fcen, fwidth=conf.inc_df),
    component=mp.Ez,
    center=conf.source_center,
    size=mp.Vector3(x=0, y=conf.cell_Y, z=conf.cell_Z),
)]

sim = mp.Simulation(
    cell_size=cell,
    boundary_layers=pml_layers,
    geometry=geometry,
    sources=sources,
    resolution=conf.resolution,
    symmetries = symmetries,
)
flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(center=conf.flux_center, size=mp.Vector3(y=conf.cell_Y, z=conf.cell_Z))
)

sim.run(until=conf.steps) # run 1st part of simulation
input_flux = mp.get_fluxes(flux_obj) # saving flux from sub in input_flux
sim.reset_meep() # purge all adjustments of simulation

### Setting up the library calculation


In [ ]:
step_for_a = 3 # Number of splittings for a
step_for_width = 50 # Number of splittings for width
sum = step_for_width * step_for_a
number_sections = 1 # Number of sections for every set of parameters

mode_tran_section = np.zeros(number_sections)
mode_phase_section = np.zeros(number_sections)
mode_tran = np.zeros(sum)
mode_phase = np.zeros(sum)

size_block = [None] * step_for_a * step_for_width  # list of mp.Vector3
min_a, max_a = 0.5, 2.5
min_width, max_width = 0.1, 3  


# curve line parameters
a = np.linspace(min_a, max_a, step_for_a) 
width = np.linspace(min_width, max_width, step_for_width) 

x_max = 5 # range x
x_min = -x_max

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

In [ ]:
width

In [ ]:
a

In [ ]:
def solve_for_x(x0, a, width):
    cosh_diff = math.cosh(x0 / a) - width / a
    if cosh_diff < 1:
        raise ValueError("Invalid value: the expression under arcosh must be >= 1")
    
    # Calculating x
    return a * math.acosh(cosh_diff)

def solve_for_max_width(width, a):
     if (width) < 0: # width must be > 0 
        raise ValueError("Invalid value: width must be > 0.")
     # Calculating x
     x = a * math.acosh(1 + width / a)
     return x        

def not_split(x_sections, a ,width):
    return (a * np.cosh(x_sections / a) - width) <= a

### Here we will calculate a big library trough MEEP
Than we will save it and put in neural network based on PyTorch

In [ ]:
n = 0 # This is a parameter for clearing the cell output, and is not related to the main calculation.
for k in range(step_for_a):
    for j in range(step_for_width):
        x_sections = np.linspace(width[j]/2, x_max, number_sections, endpoint=False, dtype=float)
        width_wide = [] # Here we collect the widths of the sections
        x_centers = [] # Here we collect their relative centers.
        mode_phase_section = np.zeros(number_sections)
        
        for i in range(number_sections):
            
            if not_split(x_sections[i], a[k], width[j]):
                x_centers.append(0)
                width_wide.append(x_sections[i]*2)
            
            else:
                find_width = x_sections[i] - solve_for_x(x_sections[i], a[k], width[j] )
                width_wide.append(find_width) 
                x_centers.append(x_sections[i] - find_width/2)
            
            geometry[1].size = mp.Vector3(x=conf.at_hei, y = width_wide[i], z = mp.inf)
            geometry[2].size = mp.Vector3(x=conf.at_hei, y = width_wide[i], z = mp.inf)
            
            if x_centers[i] == 0:   
                geometry[1].center = mp.Vector3(x = conf.at_center[0], y = 0, z = 0)
                geometry[2].center = mp.Vector3(x = conf.at_center[0], y = 0, z = 0)
            else:
                geometry[1].center = mp.Vector3(x = conf.at_center[0], y = x_centers[i], z = 0)
                geometry[2].center = mp.Vector3(x = conf.at_center[0], y = -x_centers[i], z = 0)
            
            sim = mp.Simulation(
                cell_size=cell,
                boundary_layers=pml_layers,
                geometry=geometry,
                sources=sources,
                resolution=conf.resolution,
            )
            flux_obj = sim.add_flux(
                conf.inc_fcen, 0, 1, mp.FluxRegion(
                    center=mp.Vector3(x=conf.flux_center[0]), 
                    size=mp.Vector3(y=conf.cell_Y)
                )
            )
            
            sim.run(until=conf.steps)
            
            res = sim.get_eigenmode_coefficients(
                flux_obj, [1], eig_parity=mp.ODD_Z + mp.EVEN_Y
            )
            coeffs = res.alpha
            
            mode_tran_section[i] = abs(coeffs[0, 0, 0]) ** 2 / input_flux[0]
            mode_phase_section[i] = np.angle(coeffs[0, 0, 0])
            if mode_phase_section[i] > 0:
                mode_phase_section[i] -= 2 * np.pi
                     
        mode_tran[n] = np.mean(mode_tran_section)
        mode_phase[n] = np.mean(mode_phase_section)
        print(f'step number: {n}')
        n = n+1
        if n % 20 == 0:  # Clear the output every 20 iterations
            clear_output(wait=True)

        

In [ ]:
mode_phase_section

In [ ]:
mode_phase

In [ ]:
mode_tran

In [ ]:
mode_phase.shape

In [ ]:
mode_phase.ndim

In [ ]:
print(f'first 10 curves have coefficient WIDTH: {width[0]:0.2f}')
print(f'first 10 curves have coefficient A: {a[0]:0.2f}')

In [ ]:
# Plot
plt.figure(figsize=(8, 6))  
A = list(range(1, len(mode_phase) + 1))
plt.plot(A, mode_phase, marker='o', linestyle='-', color='b', label='A vs mode_phase')

plt.title('Зависимость A от mode_phase', fontsize=14)
plt.xlabel('A', fontsize=12)
plt.ylabel('mode_phase', fontsize=12)
plt.grid(True) 
plt.legend()  
plt.tight_layout() 

plt.show()

### Fine, now we want to save result as database

In [ ]:
print(mode_tran.dtype)
print(mode_tran.ndim)

In [ ]:
# Combining it into one two-dimensional array
data = np.column_stack((mode_phase, mode_tran, width, a))

# Save it to a CSV file
np.savetxt('vectors.csv', data, delimiter=',', header='mode_phase,mode_tran,width,a', comments='', fmt='%.6f')

# Alternative: save in binary format .npy
np.save('vectors.npy', data)

print("Data saved successfully!")

In [ ]:
data.ndim

In [ ]:
data.size

In [ ]:
data.dtype

Finally. The verification is completed. The training data is calculated and stored correctly.
### TODO: Implement multithreading through parallelism of independent processes.

We have a calculated data set. But they still need to be prepared for training.
It is necessary to divide the data set into a test and a train, as well as to understand the indication of the trained parameters.